In [1]:
from pathlib import Path
import sys

# Add notebook directory so reactor_parameters can be found
_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
_nb_dir = next((p for p in _candidates if (p / "reactor_parameters.py").is_file()), None)
if _nb_dir is not None and str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import numpy as np
import matplotlib.pyplot as plt

from reactor_parameters import *

# Reproduce Lump case (no He3, He4)

In [ ]:
I_target_T_kg = 0.5  # 500 g
# Target Tritium inventory in number of atoms
N_target_T_atoms = I_target_T_kg / species_mass["T"]

### Reactivity lookup

In [ ]:
# ----- Reactivity setup -----

reactivities_cmp_full = compute_reactivities_from_functions(float(T_i))
sigmav_DD_p_cmp = float(reactivities_cmp_full["sigmav_DD_p"])
sigmav_DD_n_cmp = float(reactivities_cmp_full["sigmav_DD_n"])
sigmav_DT_cmp = float(reactivities_cmp_full["sigmav_DT"])

### Lump dd startup model

In [ ]:
# Lump model (steady-state DD approximation)
n_T_lump, n_D_lump, n_He3_lump, t_startup_lump, ok_lump = lump_numba(
    float(V_plasma),
    float(n_tot),
    float(tau_p_T),
    float(tau_p_T),
    float(TBR_DT),
    float(TBR_DD),
    float(I_target_T_kg),
    float(sigmav_DD_p_cmp),
    float(sigmav_DD_n_cmp),
    float(sigmav_DT_cmp),
    0.0,
)

### Multispecies equivalent set up to match Lump case

In [ ]:
# Multispecies equivalent: DD operation with only D/T active and T accumulation in storage.
species_params_cmp = {
    "D": {
        "f_0": 1.0,
        "tau_p": tau_p_T,
        "tau_ifc": 4.0 * 3600.0,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": np.inf,
        "N_ofc_0": float(SPECIES_DEFAULTS["D"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["D"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["D"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
    "T": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": 4.0 * 3600.0,
        "tau_ofc": 2.0 * 3600.0,
        "N_stor_min": 0.0,
        "Ndot_max": 0.0,
        "N_ofc_0": float(SPECIES_DEFAULTS["T"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["T"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["T"]["N_stor_0"]),
        "enable_plasma_channel": True,
    },
    "He3": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": np.inf,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": 0.0,
        "N_ofc_0": float(SPECIES_DEFAULTS["He3"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["He3"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["He3"]["N_stor_0"]),
        "enable_plasma_channel": False,
    },
    "He4": {
        "f_0": 0.0,
        "tau_p": tau_p_T,
        "tau_ifc": np.inf,
        "tau_ofc": np.inf,
        "N_stor_min": 0.0,
        "Ndot_max": 0.0,
        "N_ofc_0": float(SPECIES_DEFAULTS["He4"]["N_ofc_0"]),
        "N_ifc_0": float(SPECIES_DEFAULTS["He4"]["N_ifc_0"]),
        "N_stor_0": float(SPECIES_DEFAULTS["He4"]["N_stor_0"]),
        "enable_plasma_channel": False,
    },
}

inject_from_storage_cmp = {"D": False, "T": True, "He3": False, "He4": False}
injection_mode_cmp = {"D": "auto", "T": "off", "He3": "off", "He4": "off"}

species_params = {
    sp: {
        "tau_p": float(species_params_cmp[sp]["tau_p"]),
        "lambda_decay": float(species_params_cmp[sp].get("lambda_decay", SPECIES_DEFAULTS[sp]["lambda_decay"])),
        "tau_ifc": float(species_params_cmp[sp]["tau_ifc"]),
        "tau_ofc": float(species_params_cmp[sp]["tau_ofc"]),
        "N_stor_min": float(species_params_cmp[sp]["N_stor_min"]),
        "Ndot_max": float(species_params_cmp[sp]["Ndot_max"]),
        "inject_from_storage": bool(species_params_cmp[sp].get("inject_from_storage", inject_from_storage_cmp[sp])),
        "injection_mode": str(species_params_cmp[sp].get("injection_mode", injection_mode_cmp[sp])),
        "enable_plasma_channel": bool(species_params_cmp[sp]["enable_plasma_channel"]),
    }
    for sp in SPECIES
}

initial_conditions = {
    sp: {
        "f_0": float(species_params_cmp[sp]["f_0"]),
        "N_ofc_0": float(species_params_cmp[sp]["N_ofc_0"]),
        "N_ifc_0": float(species_params_cmp[sp]["N_ifc_0"]),
        "N_stor_0": float(species_params_cmp[sp]["N_stor_0"]),
    }
    for sp in SPECIES
}

In [ ]:
# Solver dictionaries are defined above.

In [ ]:


reactivities_cmp = {
    "sigmav_DD_p": sigmav_DD_p_cmp,
    "sigmav_DD_n": sigmav_DD_n_cmp,
    "sigmav_DT": sigmav_DT_cmp,
    "sigmav_DHe3": 0.0,
    "sigmav_TT": 0.0,
    "sigmav_He3He3": 0.0,
    "sigmav_THe3_ch1": 0.0,
    "sigmav_THe3_ch2": 0.0,
    "sigmav_THe3_ch3": 0.0,
}



In [5]:
# ----- Solve -----
res_ms_cmp = solve_multispecies_ode_system(
    V_plasma=float(V_plasma),
    T_i=float(T_i),
    n_tot=float(n_tot),
    species_params=species_params,
    initial_conditions=initial_conditions,
    TBR_DT=float(TBR_DT),
    TBR_DDn=float(TBR_DD),
    max_simulation_time=float(max_simulation_time),
    vector_length=int(vector_length),
    reactivities=reactivities_cmp,
    target_conditions=[{"target_specie": "T", "metric": "stor", "value": float(N_target_T_atoms)}],
)

print("Lump vs multispecies: DD operation with D/T only, target T storage = 500 g")
print(f"  Lump success = {bool(ok_lump)}")
if np.isfinite(t_startup_lump):
    print(f"  Lump t_startup = {t_startup_lump:.6e} s ({t_startup_lump/86400.0:.3f} d)")
else:
    print("  Lump t_startup = inf")

print(f"  Multispecies success = {res_ms_cmp['sol_success']}")
if np.isfinite(res_ms_cmp["t_startup"]):
    print(
        f"  Multispecies t_startup = {res_ms_cmp['t_startup']:.6e} s "
        f"({res_ms_cmp['t_startup']/86400.0:.3f} d)"
    )
else:
    print("  Multispecies t_startup = inf")

if np.isfinite(t_startup_lump) and np.isfinite(res_ms_cmp["t_startup"]):
    rel = (res_ms_cmp["t_startup"] - t_startup_lump) / t_startup_lump
    print(f"  Relative difference (multispecies vs lump) = {100.0 * rel:.2f}%")



Lump vs multispecies: DD operation with D/T only, target T storage = 500 g
  Lump success = True
  Lump t_startup = 2.201235e+07 s (254.773 d)
  Multispecies success = True
  Multispecies t_startup = 2.203570e+07 s (255.043 d)
  Relative difference (multispecies vs lump) = 0.11%


In [ ]:
t_ref = np.asarray(res_ms_cmp["t"], dtype=float)
n_D = np.maximum(np.asarray(res_ms_cmp.get("n_D", np.zeros_like(t_ref)), dtype=float), 0.0)
n_T = np.maximum(np.asarray(res_ms_cmp.get("n_T", np.zeros_like(t_ref)), dtype=float), 0.0)
n_He3 = np.maximum(np.asarray(res_ms_cmp.get("n_He3", np.zeros_like(t_ref)), dtype=float), 0.0)

(
    P_DDn,
    P_DDp,
    P_DT,
    P_DHe3,
    P_TT,
    P_He3He3,
    P_THe3_ch1,
    P_THe3_ch2,
    P_THe3_ch3,
    P_DT_eq,
) = compute_fusion_power_profiles_numba(
    n_D,
    n_T,
    n_He3,
    float(n_tot),
    float(V_plasma),
    float(reactivities_cmp["sigmav_DD_p"]),
    float(reactivities_cmp["sigmav_DD_n"]),
    float(reactivities_cmp["sigmav_DT"]),
    float(reactivities_cmp["sigmav_DHe3"]),
    float(reactivities_cmp["sigmav_TT"]),
    float(reactivities_cmp["sigmav_He3He3"]),
    float(reactivities_cmp["sigmav_THe3_ch1"]),
    float(reactivities_cmp["sigmav_THe3_ch2"]),
    float(reactivities_cmp["sigmav_THe3_ch3"]),
)

P_fusion_total = P_DDn + P_DDp + P_DT + P_DHe3 + P_TT + P_He3He3 + P_THe3_ch1 + P_THe3_ch2 + P_THe3_ch3
P_DDn = np.asarray(P_DDn, dtype=float)
P_DDp = np.asarray(P_DDp, dtype=float)
P_DT = np.asarray(P_DT, dtype=float)
P_DHe3 = np.asarray(P_DHe3, dtype=float)
P_TT = np.asarray(P_TT, dtype=float)
P_He3He3 = np.asarray(P_He3He3, dtype=float)
P_THe3_ch1 = np.asarray(P_THe3_ch1, dtype=float)
P_THe3_ch2 = np.asarray(P_THe3_ch2, dtype=float)
P_THe3_ch3 = np.asarray(P_THe3_ch3, dtype=float)
P_DT_eq = np.asarray(P_DT_eq, dtype=float)
P_fusion_total = np.asarray(P_fusion_total, dtype=float)


In [ ]:
# Plasma fractions comparison
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_ms_cmp["t"], dtype=float) / 86400.0
for sp in ["D", "T"]:
    key = f"n_{sp}"
    if key in res_ms_cmp:
        ax.plot(t_days, np.asarray(res_ms_cmp[key], dtype=float) / n_tot, label=f"f_{sp}")
total = np.zeros_like(t_days, dtype=float)
for sp in SPECIES:
    key = f"n_{sp}"
    if key in res_ms_cmp:
        total = total + np.asarray(res_ms_cmp[key], dtype=float)
ax.plot(t_days, total / n_tot, lw=2.2, ls='-.', label="f_total")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Fraction (-)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# T storage inventory
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_ms_cmp["t"], dtype=float) / 86400.0
ax.plot(t_days, np.asarray(res_ms_cmp["N_stor_T"], dtype=float) * species_mass["T"], label="I_STOR_T")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Inventory (kg)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# T injection rate
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_ms_cmp["t"], dtype=float) / 86400.0
ax.plot(t_days, np.asarray(res_ms_cmp["Ndot_inj_T"], dtype=float), label="Ndot_inj_T")
ax.set_xlabel("Time (days)")
ax.set_ylabel("Injection rate (atoms/s)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fusion powers
fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
t_days = np.asarray(res_ms_cmp["t"], dtype=float) / 86400.0
reaction_series = [
    ("P_DDn", P_DDn),
    ("P_DDp", P_DDp),
    ("P_DT", P_DT),
    ("P_fusion_total", P_fusion_total),
]
for label, series in reaction_series:
    ax.plot(t_days, np.asarray(series, dtype=float) * 1.0e-6, label=label)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Power (MW)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
# Compare storage trajectories in kg.
t_days = np.asarray(res_ms_cmp["t"]) / 86400.0
N_stor_T_ms_kg = np.asarray(res_ms_cmp["N_stor_T"]) * species_mass["T"]

N_stor_T_lump_kg = None
if np.isfinite(t_startup_lump) and t_startup_lump > 0.0:
    Tdot_tot_lump = N_target_T_atoms * lambda_T / (1.0 - np.exp(-lambda_T * t_startup_lump))
    N_stor_T_lump_kg = (Tdot_tot_lump / lambda_T) * (1.0 - np.exp(-lambda_T * np.asarray(res_ms_cmp["t"])))
    N_stor_T_lump_kg = N_stor_T_lump_kg * species_mass["T"]

fig, ax = plt.subplots(1, 1, figsize=(9, 4.5))
ax.plot(t_days, N_stor_T_ms_kg, lw=2.2, label="Multispecies N_stor,T")
if N_stor_T_lump_kg is not None:
    ax.plot(t_days, N_stor_T_lump_kg, "--", lw=2.0, label="Lump equivalent N_stor,T")
ax.axhline(I_target_T_kg, color="gray", linestyle=":", label="target = 0.5 kg")
if np.isfinite(t_startup_lump):
    ax.axvline(t_startup_lump / 86400.0, color="k", linestyle="--", alpha=0.7, label="lump t_startup")
if np.isfinite(res_ms_cmp["t_startup"]):
    ax.axvline(res_ms_cmp["t_startup"] / 86400.0, color="tab:green", linestyle="--", alpha=0.7, label="multispecies t_startup")
ax.set_xlabel("Time (days)")
ax.set_ylabel("T storage inventory (kg)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
